# Mountain-range topography: polar triplets, lapse rates, MAD

Per-range elevation x aspect structure of runoff onset: lapse rates by range, the polar triplets of
every range, MAD by range. Reads the mountain-range cube
written by `0_aggregate_by_mountain_range.ipynb`; the per-range metrics table comes from
`0_aggregate_by_mountain_range.ipynb`. Setup cells first, then any section.

The polar-triplet world map, its per-range triplet sweep and the legend are built by
`topography_triplet_composite_figure.ipynb` (one notebook per composite figure since 2026-09).

In [ ]:
import textwrap

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

from gsro_analysis import aggregate, paths, plotting, settings
from gsro_analysis.plotting import major_tick_radii

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE

# GMBA Inventory v2.0 standard 300 polygons (the cube's units), read straight from EarthEnv
gmba_gdf = gpd.read_file('zip+' + settings.GMBA_URL)
gmba_gdf

In [ ]:
# the mountain-range cube written by 0_aggregate_by_mountain_range.ipynb: range x elevation x aspect x chili_class x
# water_year, bin means as plain variables (<var>, <var>_std, <var>_n) plus the per-range ERA5-Land anomaly zonal means;
# fcf_lte_50 = the analyses' pixel filter
mountain_ranges_path = paths.aggregation_dir('mountain_ranges', config.version) / 'all_mountain_ranges_fcf_lte_50.nc'
mountain_ranges_cube_ds = xr.open_dataset(mountain_ranges_path)
mountain_ranges_cube_ds

In [ ]:
# --- the analyses' view of the cube, the same rules in every mountain-range notebook -----------------------------
MIN_PIXELS_PER_BIN = 100          # a (range, elevation, aspect) bin with this many pixels or fewer is masked
MIN_YEAR_FRACTION_OF_BIN = 0.3    # a bin-year keeps its value only with more than 30 % of the bin's median pixels
MIN_YEAR_FRACTION_OF_RANGE = 0.1  # a range-year mean anomaly needs at least 10 % of the range's median pixels

mountain_ranges_ds = aggregate.collapse(mountain_ranges_cube_ds)              # CHILI classes folded (count-weighted, exact)
mountain_ranges_ds = aggregate.threshold(mountain_ranges_ds, MIN_PIXELS_PER_BIN + 1)
enough_pixels_this_year = mountain_ranges_ds['runoff_onset_n'] > MIN_YEAR_FRACTION_OF_BIN * mountain_ranges_ds['runoff_onset_median_n']
for var in ['runoff_onset', 'runoff_onset_anomaly', 'runoff_onset_std', 'runoff_onset_anomaly_std']:
    mountain_ranges_ds[var] = mountain_ranges_ds[var].where(enough_pixels_this_year)

# tropical-Andes rule: in South American ranges north of 20 S, median onsets of 250 DOWY or later below 5000 m are
# late-season artefacts
tropical_andes = (mountain_ranges_ds['continent'] == 'South America') & (mountain_ranges_ds['centroid_latitude'] > -20)
keep = (mountain_ranges_ds['runoff_onset_median'] < 250) | (mountain_ranges_ds['elevation'] > 5000) | ~tropical_andes
for var in ['runoff_onset_median', 'runoff_onset_median_std', 'runoff_onset_mad', 'runoff_onset_mad_std',
            'runoff_onset', 'runoff_onset_anomaly', 'runoff_onset_std', 'runoff_onset_anomaly_std']:
    mountain_ranges_ds[var] = mountain_ranges_ds[var].where(keep)

# each aspect's deviation from the per-elevation median across aspects (the triplets' third panel), masked where an
# aspect is missing
elev_relative_da = aggregate.elevation_relative(mountain_ranges_ds['runoff_onset_median'])
mountain_ranges_ds['runoff_onset_elev_relative'] = elev_relative_da.where(~elev_relative_da.isnull().any('aspect'))

# the pixel-weighted mean onset anomaly per range and water year; a range-year with fewer valid pixels than
# MIN_YEAR_FRACTION_OF_RANGE of the range's median pixels is masked (the same rule as the metrics table)
range_mean_anomaly_da = aggregate.weighted_mean(mountain_ranges_ds, 'runoff_onset_anomaly', ['elevation', 'aspect'])
valid_fraction_da = mountain_ranges_ds['runoff_onset_n'].sum(['elevation', 'aspect']) / mountain_ranges_ds['runoff_onset_median_n'].sum(['elevation', 'aspect'])
mountain_ranges_ds['runoff_onset_mean_anomaly'] = range_mean_anomaly_da.where(valid_fraction_da >= MIN_YEAR_FRACTION_OF_RANGE)

# ranges without any data left are dropped
has_data = mountain_ranges_ds['runoff_onset_median'].notnull().any(('elevation', 'aspect'))
mountain_ranges_ds = mountain_ranges_ds.sel(mountain_range=has_data)
mountain_ranges_ds

In [ ]:
# polar axes want radians; the cube stores aspect in degrees
mountain_ranges_ds = mountain_ranges_ds.assign_coords(aspect=np.deg2rad(mountain_ranges_ds['aspect']))
mountain_ranges_ds

## Single-range triplet

In [ ]:
mountain_params = {
    'Olympic Mountains': {
        'runoff_cmap_min': 80,
        'runoff_cmap_max': 220,
        'bottom_r': 2000,
        'top_r': 0
    },
    'Sierra Nevada': {
        'runoff_cmap_min': 150,
        'runoff_cmap_max': 250,
        'bottom_r': 4000,
        'top_r': 0
    },
    'Cascade Range': {
        'runoff_cmap_min': 120,
        'runoff_cmap_max': 220,
        'bottom_r': 3000,
        'top_r': 0
    },
    'Brooks Range': {
        'runoff_cmap_min': 200,
        'runoff_cmap_max': 275,
        'bottom_r': 3000,
        'top_r': 0
    },
    'South-Central Alaska': {
        'runoff_cmap_min': 180,
        'runoff_cmap_max': 300,
        'bottom_r': 3000,
        'top_r': 0
    },
    'Oregon Coast Range': {
        'runoff_cmap_min': 100,
        'runoff_cmap_max': 160,
        'bottom_r': 2000,
        'top_r': 0
    },
    'Great Basin Ranges': {
        'runoff_cmap_min': 100,
        'runoff_cmap_max': 330,
        'bottom_r': 4000,
        'top_r': 1000
    }
}

In [ ]:
location = 'Sierra Nevada'
location = 'Cordillera Occidental (Central Andes)'
location = 'Cordillera Oriental (Central Andes)'
location = 'Cordillera Central (Central Andes)' # THIS ONE MOST INTERESTING
if location not in mountain_ranges_ds.mountain_range.values:   # partially processed version: fall back to a range that exists
    location = str(mountain_ranges_ds.mountain_range.values[0])
#location = 'Cordillera Occidental (Northern Andes)'
#location = 'Cordillera Oriental (Northern Andes)'
#location = 'Cordillera Central (Northern Andes)'


params = mountain_params.get(location, {
    'runoff_cmap_min': 200,
    'runoff_cmap_max': 275,
    'bottom_r': 8000,
    'top_r': 0
})

runoff_cmap_min = params['runoff_cmap_min']
runoff_cmap_max = params['runoff_cmap_max']
bottom_r = params['bottom_r']
top_r = params['top_r']

In [ ]:
f,axs=plt.subplots(1,3,figsize=(12,5.5),subplot_kw=dict(projection='polar'),dpi=300)
mountain_ranges_ds.sel(mountain_range=location)['runoff_onset_median'].plot(ax=axs[0],yincrease=False, cbar_kwargs={"label": "[DOWY]", "orientation": "horizontal"},robust=True,edgecolors='face')
mountain_ranges_ds.sel(mountain_range=location)['runoff_onset_mad'].plot(ax=axs[1],cmap='Reds',yincrease=False,cbar_kwargs={"label": "[Days]", "orientation": "horizontal"},vmin=0,vmax=30)
mountain_ranges_ds.sel(mountain_range=location)['runoff_onset_elev_relative'].plot(ax=axs[2],cmap='PuOr',yincrease=False,cbar_kwargs={"label": "[Days]", "orientation": "horizontal"},vmin=-40,vmax=40)

axs[0].set_title('10-year median runoff onset')
axs[1].set_title('10-year MAD of runoff onset')
axs[2].set_title('Runoff onset difference\nfrom elevation median')



for ax in axs.flat:
    plotting.style_polar_axes(ax, bottom_r=bottom_r, top_r=top_r, aspect_labels=True, elevation_labels=True)

f.suptitle(f'{location}', fontsize=16, y=1)

f.tight_layout(w_pad=0,h_pad=2)

## All ranges, three continent groups (the per-range triplets at a glance)

In [ ]:
def ranges_north_to_south(ds, continents):
    """Range names in the given continents, north to south (empty for a partially processed version)."""
    in_continents = ds['mountain_range'][ds['continent'].isin(continents).values]
    if in_continents.size == 0:
        return np.array([], dtype=str)
    return ds.sel(mountain_range=in_continents).sortby('centroid_latitude', ascending=False)['mountain_range'].values

americas = ranges_north_to_south(mountain_ranges_ds, ['North America', 'South America'])
euraf = ranges_north_to_south(mountain_ranges_ds, ['Europe', 'Africa'])
asoc = ranges_north_to_south(mountain_ranges_ds, ['Asia', 'Oceania'])

# Setup figure
max_ranges = max(len(americas), len(euraf), len(asoc))
fig, axs = plt.subplots(max_ranges, 9, 
                        figsize=(15, max_ranges*1.0),
                        subplot_kw={'projection': 'polar'},
                        layout='compressed',
                        dpi=300,
                        )

# Define metrics
metrics = [
    ('runoff_onset_median', 'viridis', 100, 300),
    ('runoff_onset_mad', 'Reds', 0, 30),
    ('runoff_onset_elev_relative', 'RdBu', -15, 15)
]

cbar_labels = ['10-year median runoff onset [DOWY]','10-year MAD of runoff onset [days]','Runoff onset difference from elevation median [days]']
# Plot data
for group_idx, ranges in enumerate([americas, euraf, asoc]):
    for row, mountain in enumerate(ranges):
        for metric_idx, (metric, cmap, vmin, vmax) in enumerate(metrics):
            col = group_idx*3 + metric_idx
            ax = axs[row, col]
            
            # Get data and elevation limits
            data = mountain_ranges_ds.sel(mountain_range=mountain)
            try:
                r_min = data['runoff_onset_median'].where(lambda x: x>0,drop=True).elevation.min()
                r_max = data['runoff_onset_median'].where(lambda x: x>0,drop=True).elevation.max()
            except:
                r_min = 0
                r_max = 8000

            if r_min < 1000: r_min = 0
            if r_max > 7000: r_max = 8000

            r_min = 0
            r_max = 8000
            
            # Plot data
            data[metric].plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, 
                            add_colorbar=False, yincrease=False)
            
            # Customize plot
            plotting.style_polar_axes(ax, bottom_r=r_max, top_r=r_min, center_dot=False)
            ax.set_title('')
            
            # Add mountain names for each continent group

            wrapped_text = textwrap.fill(mountain, width=15)
            if metric_idx == 0:
                ax.text(-0.2, 0.5, wrapped_text, transform=ax.transAxes,
                       va='center', ha='right')

# Remove empty subplots
for row in range(max_ranges):
    for group_idx, ranges in enumerate([americas, euraf, asoc]):
        if row >= len(ranges):
            for i in range(3):
                fig.delaxes(axs[row, group_idx*3 + i])

# Add colorbars
for i, (metric, cmap, vmin, vmax) in enumerate(metrics):
    cax = fig.add_axes([0.11, 0.03 - (i*0.015), 0.25, 0.005])
    norm = Normalize(vmin, vmax)
    plt.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax,
                orientation='horizontal',
                label=cbar_labels[i])

#plt.subplots_adjust(left=0.1, right=0.95, bottom=0.15, top=0.95, wspace=0.1)

fig.savefig(paths.figdir('mountain_ranges', config.version) / 'polar_triplets_all_ranges.png', dpi=300)

## The polar-triplet world map — built elsewhere

The per-range triplet sweep, the triplet legend and the composite polar-triplet world map are built by [`topography_triplet_composite_figure.ipynb`](topography_triplet_composite_figure.ipynb) (one notebook per composite figure since 2026-09).

## Runoff onset lapse rates by range (sqrt-count-weighted regression over the elevation x aspect bins)

In [ ]:
# the per-range lapse rates computed by 0_aggregate_by_mountain_range.ipynb (sqrt(count)-weighted regression of the bin
# median onset on elevation over the elevation x aspect bins), read from the metrics table under the column names the
# cells below use
metrics_path = paths.resultsdir('mountain_ranges', config.version) / 'mountain_range_metrics.csv'
metrics_df = pd.read_csv(metrics_path)
lapse_rates_df = metrics_df[['name', 'lapse_rate_weighted_bins_per_100m', 'lapse_rate_weighted_bins_r2', 'continent',
                             'centroid_latitude', 'centroid_longitude']].rename(columns={
    'lapse_rate_weighted_bins_per_100m': 'lapse_rate', 'lapse_rate_weighted_bins_r2': 'r_squared',
    'continent': 'CONTINENT', 'centroid_latitude': 'latitude', 'centroid_longitude': 'longitude'})
lapse_rates_df.describe()

In [ ]:
# ranges with a lapse rate, north to south; bar opacity = weighted R2 (at least 0.2)
plot_data = lapse_rates_df.dropna(subset=['lapse_rate']).sort_values('latitude', ascending=False)
alphas = plot_data['r_squared'].fillna(0.2).clip(0.2, 1)

# Define colors for continents
continent_colors = {
    'North America': '#1f77b4',
    'South America': '#ff7f0e',
    'Europe': '#2ca02c',
    'Asia': '#d62728',
    'Africa': '#9467bd',
    'Oceania': '#8c564b',
    'Antarctica': '#17becf',   # one range: South Atlantic Islands
}

# Create plot
f, ax = plt.subplots(figsize=(15, 25))

# Create bars with single alpha value
bars = ax.barh(y=range(len(plot_data)), 
               width=plot_data['lapse_rate'],
               color=[continent_colors.get(c, '#7f7f7f') for c in plot_data['CONTINENT']],
               alpha=0.7)  # Single alpha value


for bar, alpha in zip(bars, alphas):
    bar.set_alpha(alpha)
    
# Add range names to y-axis
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data['name'])

# Add styling
ax.axvline(x=0, color='black', linestyle='--', alpha=0.3)
ax.grid(True, alpha=0.3)
ax.set_xlabel('Days delay per 100m elevation gain')
ax.set_ylabel('Mountain Range (ordered by latitude)')
ax.set_title('Runoff Onset Lapse Rates by Mountain Range')

# Add legend
legend_elements = [plt.Rectangle((0,0),1,1, facecolor=color, label=cont) 
                  for cont, color in continent_colors.items()]
ax.legend(handles=legend_elements, 
         title='Continent', 
         bbox_to_anchor=(1.05, 1), 
         loc='upper left')

ax.invert_yaxis()

f.tight_layout()

f.savefig(paths.figdir('mountain_ranges', config.version) / 'global_lapse_rates_by_mountain_range.png',dpi=300)

In [ ]:
# Create dictionary mapping from names to GMBA_V2_ID
name_to_id = {}
for idx, row in gmba_gdf.iterrows():
    if row['MapName'] in lapse_rates_df['name'].values:
        name_to_id[row['MapName']] = row['GMBA_V2_ID']
    elif row['Level_04'] in lapse_rates_df['name'].values:
        name_to_id[row['Level_04']] = row['GMBA_V2_ID']

# Add GMBA_V2_ID to lapse_rates_df
lapse_rates_df['GMBA_V2_ID'] = lapse_rates_df['name'].map(name_to_id)

# Merge with GMBA geometries
lapse_rates_gdf = lapse_rates_df.merge(
    gmba_gdf[['GMBA_V2_ID', 'geometry']], 
    on='GMBA_V2_ID', 
    how='left'
)

# Convert to GeoDataFrame
lapse_rates_gdf = gpd.GeoDataFrame(
    lapse_rates_gdf, 
    geometry='geometry',
    crs=gmba_gdf.crs
)

# Sort by continent and latitude
lapse_rates_gdf = lapse_rates_gdf.sort_values(['CONTINENT', 'latitude'], ascending=[True, False])
lapse_rates_gdf

In [ ]:
f,ax=plt.subplots(figsize=(20,10),subplot_kw=dict(projection=ccrs.EqualEarth()),dpi=300)
plots = lapse_rates_gdf.plot(ax=ax,column='r_squared',transform=ccrs.PlateCarree(),cmap='inferno',legend=True)
#f.colorbar(plots, ax=ax, orientation='vertical', label='Lapse Rate')
ax.set_title('') 

gl = ax.gridlines(draw_labels=True, 
                  dms=True, 
                  x_inline=False, 
                  y_inline=False, 
                  xlocs=[-180, -120, -60, 0, 60, 120, 180], 
                  ylocs=[-60, -40, -20, 0, 20, 40, 60, 80], 
                  linestyle='--', 
                  linewidth=0.5,
                  auto_update=True,
                  )



gl.top_labels=False
gl.bottom_labels=True
gl.right_labels=False
gl.left_labels=True


ax.add_feature(cfeature.LAND, facecolor='lightgrey')
ax.add_feature(cfeature.OCEAN, facecolor='powderblue') 
ax.coastlines(resolution='110m', linewidth=0.5, edgecolor='black')

ax.set_extent([-200, 200, -62, 82], crs=ccrs.PlateCarree())
ax.set_xlim(left=-1.25E7)

In [ ]:
# Create figure with subplot mosaic
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(2, 2, width_ratios=[1.5, 1])

# World map with points
ax_map = fig.add_subplot(gs[:, 0], projection=ccrs.Robinson())
ax_scatter = fig.add_subplot(gs[0, 1])
ax_bars = fig.add_subplot(gs[1, 1])

# Plot world map
ax_map.add_feature(cfeature.LAND, facecolor='lightgray')
ax_map.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax_map.add_feature(cfeature.COASTLINE)

# Color scheme
continent_colors = {'North America': '#1f77b4', 'South America': '#ff7f0e', 
                   'Europe': '#2ca02c', 'Asia': '#d62728', 'Africa': '#9467bd', 'Oceania': '#8c564b', 'Antarctica': '#17becf'}

# Plot mountain ranges on map
for continent in continent_colors:
    mask = lapse_rates_df['CONTINENT'] == continent
    data = lapse_rates_df[mask]
    
    sc = ax_map.scatter(data['longitude'], data['latitude'], 
                       c=data['lapse_rate'],
                       s=data['r_squared']*300,
                       transform=ccrs.PlateCarree(),
                       cmap='RdYlBu',
                       label=continent)

# Scatter plot: Latitude vs Lapse Rate
for continent in continent_colors:
    mask = lapse_rates_df['CONTINENT'] == continent
    data = lapse_rates_df[mask]
    
    ax_scatter.scatter(data['latitude'], data['lapse_rate'],
                      c=continent_colors.get(continent, '#7f7f7f'),
                      s=data['r_squared']*100,
                      alpha=0.6,
                      label=continent)

# Horizontal bar plot
continents = []
ranges = []
rates = []
colors = []

for continent in continent_colors:
    mask = lapse_rates_df['CONTINENT'] == continent
    data = lapse_rates_df[mask].sort_values('latitude', ascending=False)
    
    continents.extend([continent] * len(data))
    ranges.extend(data['name'])
    rates.extend(data['lapse_rate'])
    colors.extend([continent_colors.get(continent, '#7f7f7f')] * len(data))

y_pos = np.arange(len(ranges))
ax_bars.barh(y_pos, rates, color=colors, alpha=0.6)

# Styling
ax_map.set_title('Mountain Range Lapse Rates (days/100m)')
ax_scatter.set_title('Lapse Rate vs Latitude')
ax_scatter.set_xlabel('Latitude')
ax_scatter.set_ylabel('Days delay per 100m')
ax_scatter.grid(True, alpha=0.3)
ax_scatter.legend()

ax_bars.set_yticks(y_pos)
ax_bars.set_yticklabels(ranges)
ax_bars.set_xlabel('Days delay per 100m')
ax_bars.axvline(x=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()


## MAD by range

In [ ]:
# pixel-weighted mean MAD per range (the old count-weighted sum over aspect and elevation)
average_mad_da = aggregate.weighted_mean(mountain_ranges_ds, 'runoff_onset_mad', ['aspect', 'elevation'])
range_metadata = pd.DataFrame({
    'name': mountain_ranges_ds.mountain_range.values,
    'CONTINENT': mountain_ranges_ds.continent.values,
    'latitude': mountain_ranges_ds.centroid_latitude.values,
    'longitude': mountain_ranges_ds.centroid_longitude.values,
})
average_mad_da

In [ ]:
# Process data
mad_data = pd.DataFrame({
    'name': average_mad_da.mountain_range.values,
    'mad': average_mad_da.values
})

# Merge with metadata to get continent information
mad_data = mad_data.merge(range_metadata[['name', 'CONTINENT', 'latitude']], on='name')
plot_data = mad_data.sort_values('latitude', ascending=False)

# Define colors for continents (same as before)
continent_colors = {
    'North America': '#1f77b4',
    'South America': '#ff7f0e',
    'Europe': '#2ca02c',
    'Asia': '#d62728',
    'Africa': '#9467bd',
    'Oceania': '#8c564b',
    'Antarctica': '#17becf',   # one range: South Atlantic Islands
}

# Create plot
fig, ax = plt.subplots(figsize=(12, 25))

# Create bars
bars = ax.barh(y=range(len(plot_data)), 
               width=plot_data['mad'],
               color=[continent_colors.get(c, '#7f7f7f') for c in plot_data['CONTINENT']],
               alpha=0.7)

# Add range names to y-axis
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data['name'])

# Add styling
ax.grid(True, alpha=0.3)
ax.set_xlabel('Median Absolute Deviation [days]')
ax.set_ylabel('Mountain Range (ordered by latitude)')
ax.set_title('Snowmelt Runoff Onset Timing Variability by Mountain Range')

# Add legend
legend_elements = [plt.Rectangle((0,0),1,1, facecolor=color, label=cont) 
                  for cont, color in continent_colors.items()]
ax.legend(handles=legend_elements, 
         title='Continent', 
         bbox_to_anchor=(1.05, 1), 
         loc='upper left')

ax.invert_yaxis()

fig.tight_layout()

fig.savefig(paths.figdir('mountain_ranges', config.version) / 'global_mad_by_mountain_range.png', dpi=300)